# Text-Only Regressions (384 embedding dimensions, OLS or ridge)

Same walk-forward design as `prediction_linear_regression_all_features.ipynb` -- monthly refit,
252-trading-day rolling window, daily out-of-sample predictions -- but the regressors are the
**stock-day text embedding only**: the 384 mean-pooled sentence-embedding dimensions, optionally
plus the two agreement measures (`embed_norm`, `embed_cos`) built in `build_text_master`. No
sentiment, volume or attention features, and no message count.

Three switches (environment variables override the defaults so the runner can launch variants):

| switch | values | meaning |
|---|---|---|
| `ESTIMATOR` | `ols`, `ridge` | ridge standardises the columns inside each window and shrinks; OLS is the `lambda = 0` special case of the same closed form |
| `TARGET_DEMEAN` | `True`/`False` | de-mean the target **and** the regressors by date inside each training window (date fixed effects), and de-mean the regressors by date at prediction time. The 04 evaluations de-mean by date anyway; this makes training consistent with that |
| `FEATURE_SET` | `embed`, `embed+norm` | 384 dims, or 384 + `embed_norm` + `embed_cos` |

**Ridge penalty.** `alpha = lambda * n_train` on standardised columns, `lambda` from
`LAMBDA_GRID`. Every candidate is fitted every month (one eigendecomposition of the 384x384
Gram matrix serves all of them). The `lambda` used for month *m*'s predictions is the one with
the lowest out-of-sample squared error over the previous `SELECT_MONTHS` months (walk-forward,
no look-ahead); until any history exists, `LAMBDA_DEFAULT`. The per-month choice and the
full candidate-by-month error table are saved next to the predictions.

Input: `Data/text_master.pkl`. Output: `Data/predictions_<model>_textonly[_dm]_input=<N>.pkl`
with `<model>` = `linear_regression` or `ridge`, `_dm` when `TARGET_DEMEAN`, `<N>` = number of
regressors; `index` = the `merged_master` row label. Distinct model names keep these files out of
the `find_all_features_file()` resolvers in 04/05; they are registered explicitly.


In [ ]:
# Import required packages
import json
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.linear_model import enet_path


In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "text_master.pkl"

# =============================================================================
# SWITCHES (environment variables TEXTONLY_ESTIMATOR / TEXTONLY_DEMEAN / TEXTONLY_FEATURES override)
# =============================================================================
ESTIMATOR     = os.environ.get("TEXTONLY_ESTIMATOR", "ridge")          # "ols" | "ridge" | "lasso" | "enet"
TARGET_DEMEAN = os.environ.get("TEXTONLY_DEMEAN", "1") == "1"          # date fixed effects inside each window
FEATURE_SET   = os.environ.get("TEXTONLY_FEATURES", "embed+norm")     # "embed" | "embed+norm"
RANK_TARGET   = os.environ.get("RANK_TARGET", "0") == "1"                # train on the daily percentile rank of the target (minus 0.5)
TARGET_COL    = os.environ.get("TARGET_COL", "f_cumret1")                # e.g. "ar_dgtw_1" for the DGTW-adjusted next-day return
FEATURE_SETS  = {"embed": "textonly", "embed+norm": "textonly", "core": "core", "embed+norm+core": "textcore",
                 "all": "all", "embed+norm+all": "textall"}                # all = the 53 non-text features
CORE_COLS     = ['net_sentiment', 'log_volume']                          # the 2-feature baseline's regressors, for the decomposition
TARGET_TAGS   = {"f_cumret1": "", "ar_dgtw_1": "_dgtw"}
assert ESTIMATOR in ("ols", "ridge", "lasso", "enet") and FEATURE_SET in FEATURE_SETS
assert TARGET_COL in TARGET_TAGS, f"add a short tag for {TARGET_COL} to TARGET_TAGS"

# Penalty candidates, all on standardised columns.
#   ridge:        alpha = lambda * n_train (closed form, one eigendecomposition per month)
#   lasso / enet: lambda is a FRACTION of alpha_max, the smallest penalty at which every coefficient
#                 of that month's window is zero, so the grid is comparable across months; 1.0 = null model
#   ols:          [0.0]
if ESTIMATOR == "ridge":
    LAMBDA_GRID, LAMBDA_DEFAULT = [0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0, 10000.0], 1.0
elif ESTIMATOR == "lasso":
    LAMBDA_GRID, LAMBDA_DEFAULT = [1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001], 0.1
elif ESTIMATOR == "enet":
    # Elastic net = lasso on the ridge-augmented Gram matrix (G + l2 * n * I): the L1 penalty is a
    # fraction of alpha_max as for the lasso, the L2 penalty is on the ridge scale (lambda * n).
    # (sklearn's l1_ratio parametrisation ties both to one alpha; at lasso-sized alphas the L2
    # term is then ~1e-4 of the Gram diagonal and the "elastic net" reproduces the lasso exactly.)
    ENET_L2 = [1.0, 10.0]
    ENET_L1 = [0.3, 0.1, 0.03, 0.01, 0.003]
    LAMBDA_GRID = [f"l2={l2:g}|l1={l1:g}" for l2 in ENET_L2 for l1 in ENET_L1]
    LAMBDA_DEFAULT = "l2=1|l1=0.1"
else:
    LAMBDA_GRID, LAMBDA_DEFAULT = [0.0], 0.0
ENET_MAX_ITER, ENET_TOL = 2000, 1e-4
SELECT_MONTHS  = 12     # lambda for month m is chosen on the previous SELECT_MONTHS months' OOS record (LAMBDA_DEFAULT until any history)
# Selection criterion over those months: "sse" = lowest squared error (both sides de-meaned by date,
# as the 04 notebooks score); "rankcorr" = highest mean daily rank correlation (the ordering metric).
SELECT_CRITERION = os.environ.get("TEXTONLY_SELECT", "sse")
assert SELECT_CRITERION in ("sse", "rankcorr")

TRAIN_END_DATE = '2011-12-31'
WINDOW = 252            # rolling window size in trading days
N_JOBS = 4              # threads; each month holds a float64 copy of its window (up to ~600k x 386)

MODEL_NAME = ({"ols": "linear_regression", "ridge": "ridge", "lasso": "lasso", "enet": "elasticnet"}[ESTIMATOR] + "_" + FEATURE_SETS[FEATURE_SET]
              + ("_dm" if TARGET_DEMEAN else "") + ("_rank" if RANK_TARGET else "") + TARGET_TAGS[TARGET_COL]
              + ("_rc" if SELECT_CRITERION == "rankcorr" else ""))
print(f"ESTIMATOR={ESTIMATOR}  TARGET_DEMEAN={TARGET_DEMEAN}  FEATURE_SET={FEATURE_SET}  RANK_TARGET={RANK_TARGET}  "
      f"TARGET_COL={TARGET_COL}  SELECT_CRITERION={SELECT_CRITERION}  -> model name {MODEL_NAME}")
print(f"lambda grid: {LAMBDA_GRID}")


In [ ]:
# Load data (row label = merged_master row label, so predictions can be placed back onto the panel)
data = pd.read_pickle(INPUT_DATA).set_index("mm_index")
data["date"] = pd.to_datetime(data["date"])
print(f"Loaded {len(data):,} stock-days with text, {data['date'].min().date()} to {data['date'].max().date()}")


In [ ]:
# Prepare features and target
TARGET = TARGET_COL

EMBED_COLS = [c for c in data.columns if c.startswith('embed_') and c not in ('embed_n', 'embed_norm', 'embed_cos')]
NORM_COLS = ['embed_norm', 'embed_cos']
assert len(EMBED_COLS) == 384, len(EMBED_COLS)
# the 53 non-text stock-day features carried over from merged_master (everything that is not a
# key, a return column, or an embedding-derived column)
ALL_COLS = [c for c in data.columns
            if not (c.startswith('ar_') or c.startswith('embed_') or c in ('permno', 'ticker', 'date', 'f_cumret1', 'year_month'))]
assert len(ALL_COLS) == 53, (len(ALL_COLS), ALL_COLS)
FEATURES = {"embed": EMBED_COLS,                                   # embed_n (message count) deliberately excluded
            "embed+norm": EMBED_COLS + NORM_COLS,
            "core": CORE_COLS,                                     # attention + tag only, on the same tweeted sample
            "embed+norm+core": EMBED_COLS + NORM_COLS + CORE_COLS,
            "all": ALL_COLS,                                       # the 53 features alone, on the tweeted sample
            "embed+norm+all": EMBED_COLS + NORM_COLS + ALL_COLS}[FEATURE_SET]

# Remove missing values (only the target can be missing here)
model_data = data[[TARGET] + FEATURES].dropna()
model_data[TARGET] = model_data[TARGET].astype('float64')
model_data['date'] = data.loc[model_data.index, 'date']
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')
if RANK_TARGET:
    # daily percentile rank among the rows of this table (tweeted stock-days), centred at 0
    model_data[TARGET] = model_data.groupby('date')[TARGET].rank(pct=True) - 0.5
    print(f"RANK_TARGET: {TARGET} replaced by its daily percentile rank - 0.5")

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")


# OOS Predictions (Monthly Training, Daily Predictions)

In [ ]:
# Dates and months
unique_dates = pd.DatetimeIndex(model_data['date'].unique()).sort_values()
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]
oos_months = sorted(model_data.loc[model_data['date'] > train_end, 'year_month'].unique())

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")


In [ ]:
def demean_by_date(X, dates):
    # subtract the per-date cross-sectional mean of every column (date fixed effects), in place
    # and in row chunks so that no second copy of the window's design matrix is materialised
    codes, _ = pd.factorize(dates)
    means = pd.DataFrame(X).groupby(codes).mean().to_numpy()
    for s in range(0, len(X), 100_000):
        X[s:s + 100_000] -= means[codes[s:s + 100_000]]
    return X


def demean_vec_by_date(y, dates):
    codes, _ = pd.factorize(dates)
    return y - pd.Series(y).groupby(codes).transform('mean').to_numpy()


def mean_daily_rank_corr(p, y, dates):
    # mean over the month's dates of the within-date Spearman correlation between p and y;
    # a date on which p is constant contributes 0 (no ordering information)
    codes, _ = pd.factorize(dates)
    t = pd.DataFrame({'c': codes, 'p': p, 'y': y})
    g = t.groupby('c')
    rp, ry = g['p'].rank(pct=True), g['y'].rank(pct=True)
    out = pd.DataFrame({'c': codes, 'x': rp, 'y': ry}).groupby('c').apply(lambda d: d['x'].corr(d['y']))
    return float(out.fillna(0.0).mean()) if len(out) else 0.0


def month_window(pred_month):
    # rows of the 252-day window before pred_month and the month's own prediction dates
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return None
    train_cutoff = pred_month.to_timestamp() - pd.Timedelta(days=2)
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return None
    last_idx = unique_dates.get_loc(train_dates[-1])
    if last_idx < WINDOW:
        return None
    start_date = unique_dates[last_idx - WINDOW + 1]
    return start_date, train_dates[-1], month_dates


def fit_and_predict_month(pred_month, sl):
    # sl: model_data slice covering the window and the month. Returns per-lambda predictions for
    # every row of the month plus per-lambda OOS SSE of the month (on the date-de-meaned realised target).
    w = month_window(pred_month)
    if w is None:
        return None
    start_date, last_train_date, month_dates = w
    tr = (sl['date'] >= start_date) & (sl['date'] <= last_train_date)
    te = sl['date'].isin(month_dates)
    if tr.sum() == 0 or te.sum() == 0:
        return None

    Xtr = sl.loc[tr, FEATURES].to_numpy(dtype=np.float64)
    ytr = sl.loc[tr, TARGET].to_numpy(dtype=np.float64)
    Xte = sl.loc[te, FEATURES].to_numpy(dtype=np.float64)
    yte = sl.loc[te, TARGET].to_numpy(dtype=np.float64)
    dtr, dte = sl.loc[tr, 'date'].to_numpy(), sl.loc[te, 'date'].to_numpy()
    if TARGET_DEMEAN:
        Xtr = demean_by_date(Xtr, dtr)
        ytr = demean_vec_by_date(ytr, dtr)
        Xte = demean_by_date(Xte, dte)
    yte_dm = demean_vec_by_date(yte, dte)     # evaluation-consistent realised target

    # standardise on the training window (in place); centre y
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
    sd[sd == 0] = 1.0
    Xtr -= mu
    Xtr /= sd
    Z = Xtr
    ybar = ytr.mean()
    yc = ytr - ybar
    n = len(Z)

    G = Z.T @ Z
    g = Z.T @ yc
    if ESTIMATOR in ("lasso", "enet"):
        # Coordinate-descent L1 path on the precomputed Gram matrix (O(p^2) per pass, so even the
        # 600k-row windows take seconds). L1 penalties are fractions of alpha_max, the smallest
        # penalty at which every coefficient is zero. For the elastic net the ridge part enters
        # through the Gram matrix, G + l2 * n * I (equivalent to data augmentation), so the path
        # stays a pure L1 path and sklearn's duality-gap stopping rule remains exact. sklearn sorts
        # the alphas descending and warm-starts along the path; alphas_out gives the order used.
        alpha_max = np.abs(g).max() / n
        G = np.ascontiguousarray(G)
        if ESTIMATOR == "lasso":
            runs = [(None, LAMBDA_GRID)]
        else:
            runs = [(l2, ENET_L1) for l2 in ENET_L2]
        betas = {}
        for l2, l1_grid in runs:
            Q = G if l2 is None else G + (l2 * n) * np.eye(G.shape[0])
            alphas_out, coefs, _ = enet_path(Z, yc, l1_ratio=1.0, alphas=alpha_max * np.array(l1_grid),
                                             precompute=Q, Xy=g, copy_X=False, check_input=False,
                                             max_iter=ENET_MAX_ITER, tol=ENET_TOL)
            fr = alphas_out / alpha_max
            for l1 in l1_grid:
                key = l1 if l2 is None else f"l2={l2:g}|l1={l1:g}"
                betas[key] = coefs[:, int(np.argmin(np.abs(fr - l1)))]
    else:
        # one eigendecomposition serves every lambda: beta = V diag(1/(d + lambda n)) V' Z'y
        d, V = np.linalg.eigh(G)
        Vg = V.T @ g
        betas = {}
        for lam in LAMBDA_GRID:
            denom = d + lam * n
            inv = np.where(denom > 1e-10 * denom.max(), 1.0 / np.where(denom == 0, 1.0, denom), 0.0)
            betas[lam] = V @ (Vg * inv)
    nonzero = {lam: int((b != 0).sum()) for lam, b in betas.items()}
    del Z, Xtr
    Zte = (Xte - mu) / sd
    preds, sse, rc = {}, {}, {}
    for lam in LAMBDA_GRID:
        p = Zte @ betas[lam] + ybar
        preds[lam] = p
        # OOS squared error the way the 04 notebooks score it: both sides de-meaned by date
        sse[lam] = float(((yte_dm - demean_vec_by_date(p, dte)) ** 2).sum())
        # and the month's mean daily rank correlation (the ordering metric)
        rc[lam] = mean_daily_rank_corr(p, yte_dm, dte)
    return {'month': pred_month, 'index': sl.index[te].to_numpy(), 'date': dte, 'preds': preds,
            'sse': sse, 'rc': rc, 'nonzero': nonzero, 'n_train': int(n), 'n_test': int(te.sum())}


def get_month_slice(pred_month):
    w = month_window(pred_month)
    if w is None:
        return model_data.iloc[0:0]
    start_date, _, month_dates = w
    return model_data.loc[(model_data['date'] >= start_date) & (model_data['date'] <= month_dates.max())]


In [ ]:
# Fit every month (all lambdas) in parallel
t0 = time.time()
print(f"Fitting {len(oos_months)} months x {len(LAMBDA_GRID)} lambdas with {N_JOBS} threads...")
results = Parallel(n_jobs=N_JOBS, verbose=5, backend='threading')(
    delayed(fit_and_predict_month)(pm, get_month_slice(pm)) for pm in oos_months)
results = [r for r in results if r is not None]
print(f"Done: {len(results)} months in {(time.time() - t0) / 60:.1f} min")


In [ ]:
# Walk-forward choice of lambda: for month m use the lambda with the lowest OOS SSE over the
# previous SELECT_MONTHS months (only past months enter; LAMBDA_DEFAULT until any history exists).
sse_tbl = pd.DataFrame({r['month']: r['sse'] for r in results}).T.sort_index()   # months x lambdas
sse_tbl.index.name = 'month'
rc_tbl = pd.DataFrame({r['month']: r['rc'] for r in results}).T.sort_index()     # months x lambdas
n_test = pd.Series({r['month']: r['n_test'] for r in results}).sort_index()

chosen = {}
for i, m in enumerate(sse_tbl.index):
    if len(LAMBDA_GRID) == 1:
        chosen[m] = LAMBDA_GRID[0]
        continue
    lo, hi = max(0, i - SELECT_MONTHS), i
    if hi - lo == 0:
        chosen[m] = LAMBDA_DEFAULT
    elif SELECT_CRITERION == "sse":
        chosen[m] = sse_tbl.iloc[lo:hi].sum(axis=0).idxmin()
    else:
        chosen[m] = rc_tbl.iloc[lo:hi].mean(axis=0).idxmax()
chosen = pd.Series(chosen, name='lambda')

print(f"Chosen lambda by month (walk-forward, criterion = {SELECT_CRITERION}):")
print(chosen.value_counts().sort_index().rename('months').to_string())
print("\nRealised OOS MSE (x1e4) and mean daily rank correlation by candidate lambda (whole OOS period, for information only):")
print(pd.DataFrame({'MSE_x1e4': (sse_tbl.sum(axis=0) / n_test.sum() * 1e4).round(4),
                    'rank_corr': rc_tbl.mean(axis=0).round(4)}).to_string())

nz_tbl = pd.DataFrame({r['month']: r['nonzero'] for r in results}).T.sort_index()   # months x lambdas
nz_chosen = pd.Series({m: nz_tbl.loc[m, chosen[m]] for m in nz_tbl.index})
if ESTIMATOR in ("lasso", "enet"):
    print(f"\nNon-zero coefficients at the chosen lambda: median {int(nz_chosen.median())}, "
          f"min {int(nz_chosen.min())}, max {int(nz_chosen.max())} of {len(FEATURES)}")
    print("Median non-zero coefficients by candidate lambda:")
    print(nz_tbl.median().astype(int).to_string())


In [ ]:
# Assemble the predictions with the chosen lambda per month
rows = []
for r in results:
    lam = chosen[r['month']]
    rows.append(pd.DataFrame({'date': r['date'], 'index': r['index'], 'prediction': r['preds'][lam]}))
predictions_df = pd.concat(rows, ignore_index=True)

# Add symbol information (data is indexed by the merged_master row label)
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index().rename(columns={'mm_index': 'index'}),
    on='index', how='left')
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"Prediction spread: std {predictions_df['prediction'].std():.5f}, "
      f"1st/99th pct {predictions_df['prediction'].quantile(0.01):.5f} / {predictions_df['prediction'].quantile(0.99):.5f}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))


In [ ]:
# Save predictions (+ the lambda path and the candidate error table as a sidecar)
OUTPUT_FILE = MODEL_DATA_DIR / f"predictions_{MODEL_NAME}_input={len(FEATURES)}.pkl"
predictions_df.to_pickle(OUTPUT_FILE)
side = {'estimator': ESTIMATOR, 'target_demean': TARGET_DEMEAN, 'feature_set': FEATURE_SET, 'features': len(FEATURES),
        'target': TARGET_COL, 'rank_target': RANK_TARGET, 'select_criterion': SELECT_CRITERION,
        'enet_l2_grid': ENET_L2 if ESTIMATOR == "enet" else None,
        'rank_corr_by_month_and_lambda': {str(k): {str(l): v for l, v in row.items()} for k, row in rc_tbl.iterrows()},
        'window': WINDOW, 'train_end': TRAIN_END_DATE, 'lambda_grid': LAMBDA_GRID, 'select_months': SELECT_MONTHS,
        'lambda_default': LAMBDA_DEFAULT, 'chosen_lambda_by_month': {str(k): v for k, v in chosen.items()},
        'nonzero_at_chosen_lambda_by_month': {str(k): int(v) for k, v in nz_chosen.items()},
        'sse_by_month_and_lambda': {str(k): {str(l): v for l, v in row.items()} for k, row in sse_tbl.iterrows()},
        'n_test_by_month': {str(k): int(v) for k, v in n_test.items()},
        'created': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}
OUTPUT_FILE.with_suffix('.json').write_text(json.dumps(side, indent=1))

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")
